# 05 — Goal 1: occurrence and acquisition variability

Runs participant-clustered descriptives and participant-level ALS/control contrasts.

This notebook saves its visual summary and audit tables into separate `figures/` and `tables/` directories. Its final cell states the main output, any decision required, and whether the next stage is allowed.

In [ ]:
from pathlib import Path
import json
import shutil
import subprocess
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
from IPython.display import Image, Markdown, display

def find_project_root():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "paper1_qc").exists():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the paper_1 project.")

ROOT = find_project_root()
CONFIG = ROOT / "config" / "project.yaml"
OUTPUT = ROOT / "outputs"
MAIN_OUTPUTS = ROOT / "MAIN outputs"
MAIN_OUTPUTS.mkdir(exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

def run_cli(*arguments):
    command = [sys.executable, "-m", "paper1_qc.cli", "--config", str(CONFIG), *arguments]
    print("RUN:", " ".join(map(str, command)))
    subprocess.run(command, cwd=ROOT, check=True)

def read_table(path_without_suffix):
    stem = Path(path_without_suffix)
    parquet = stem.with_suffix(".parquet")
    csv = stem.with_suffix(".csv")
    if parquet.exists():
        return pd.read_parquet(parquet)
    if csv.exists():
        try:
            return pd.read_csv(csv)
        except pd.errors.EmptyDataError:
            return pd.DataFrame()
    raise FileNotFoundError(f"Missing table: {parquet} or {csv}")

def stage_directories(relative_stage):
    stage = OUTPUT / relative_stage
    figures = stage / "figures"
    tables = stage / "tables"
    figures.mkdir(parents=True, exist_ok=True)
    tables.mkdir(parents=True, exist_ok=True)
    return stage, figures, tables

def save_table(frame, directory, name):
    path = Path(directory) / f"{name}.csv"
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(path, index=False)
    print("TABLE:", path.relative_to(ROOT), f"({len(frame):,} rows)")
    return path

def save_figure(fig, directory, name):
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    png = directory / f"{name}.png"
    svg = directory / f"{name}.svg"
    fig.savefig(png, bbox_inches="tight")
    fig.savefig(svg, bbox_inches="tight")
    print("FIGURE:", png.relative_to(ROOT))
    return png, svg

def stage_gate(stage_name, can_continue, reasons, next_step):
    status = "PASS — safe to continue" if can_continue else "BLOCKED — decision/action required"
    color = "#1B7F3A" if can_continue else "#B22222"
    details = "\n".join(f"- {reason}" for reason in reasons) if reasons else "- No blocking findings."
    display(Markdown(
        f"### {stage_name}: <span style='color:{color}'>{status}</span>\n\n"
        f"{details}\n\n**Next step:** {next_step}"
    ))
    return can_continue

assert CONFIG.exists(), "Copy config/project.example.yaml to config/project.yaml and review it."
print("Project:", ROOT)
print("Config:", CONFIG)
print("MAIN outputs:", MAIN_OUTPUTS)


## Main output and decisions

Main output: metric support/descriptives and exploratory participant-level effect sizes. Diagnosis-associated Q differences are acquisition/confounding patterns, not diagnostic biomarker performance.

In [ ]:
RUN_GOAL1 = False  # Change to True only when you intend to run this stage.

if RUN_GOAL1:
    run_cli('describe')
else:
    print('Goal 1 analysis not run.')

In [ ]:
STAGE, FIGURES, TABLES = stage_directories(Path("04_analysis") / "goal1")
descriptive = read_table(OUTPUT / "04_analysis" / "descriptive" / "metric_descriptive_statistics")
contrasts = read_table(OUTPUT / "04_analysis" / "descriptive" / "exploratory_participant_level_diagnosis_contrasts")
save_table(descriptive, TABLES, "metric_descriptive_statistics")
save_table(contrasts, TABLES, "participant_level_diagnosis_contrasts")
display(descriptive)
display(contrasts)

plot = contrasts.loc[contrasts["status"].eq("ok")].sort_values("cliffs_delta_a_vs_b")
fig, ax = plt.subplots(figsize=(10, max(5, 0.28 * len(plot))))
if not plot.empty:
    y = np.arange(len(plot))
    ax.errorbar(
        plot["cliffs_delta_a_vs_b"], y,
        xerr=[
            plot["cliffs_delta_a_vs_b"] - plot["cliffs_delta_ci_low"],
            plot["cliffs_delta_ci_high"] - plot["cliffs_delta_a_vs_b"],
        ],
        fmt="o", color="#4C78A8", ecolor="#9ECAE1", capsize=2,
    )
    ax.set_yticks(y)
    ax.set_yticklabels(plot["feature"], fontsize=8)
ax.axvline(0, color="black", linewidth=0.8)
ax.set(title="Participant-level ALS vs control Q contrasts", xlabel="Cliff's delta (ALS vs controls)")
fig.tight_layout()
save_figure(fig, FIGURES, "diagnosis_effect_size_forest")
plt.show()

blocked = contrasts["status"].ne("ok").sum() if not contrasts.empty else 0
goal1_ready = stage_gate(
    "Goal 1",
    blocked == 0,
    [f"{blocked} contrasts were under-supported or failed; retain them as explicit blocked rows."]
    if blocked else [],
    "Proceed to Goal 2; do not delete under-supported estimands.",
)